# Offline Trace Generation Validation\n\nThis notebook validates `parser -> routing -> state_init -> retrieval shaping -> operator priors -> branch_controller -> trace generation` using production `src/*` modules only. It is deterministic, exposes intermediate artifacts, records failures instead of filtering them out, and writes inspectable outputs.\n

In [ ]:
from __future__ import annotations\n\nimport csv\nimport json\nimport math\nimport random\nimport shutil\nfrom collections import Counter\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom src.branches.branch_controller import BranchController, ControllerConfig\nfrom src.branches.branch_state import BranchState\nfrom src.common.schemas import ParsedProblem, RouteDecision\nfrom src.operators.library import OperatorLibrary\nfrom src.operators.priors import OperatorPriorShaper\nfrom src.parsing.parser import ProblemParser\nfrom src.retrieval.embedder import MathEmbedder\nfrom src.retrieval.index_builder import RetrievalIndexBuilder\nfrom src.retrieval.query import TraceRetriever\nfrom src.retrieval.retrieval_policy import RetrievalPolicy\nfrom src.routing.dual_router import DualRouter\nfrom src.state_graph.state_init import StateGraphInitializer\n\nSEED = 1337\nSAMPLE_SIZE = 32\nOVERWRITE_OUTPUTS = True\n\nROOT = Path.cwd()\nDATASET_PATH = ROOT / 'data' / 'raw' / 'aimo3_train.csv'\nINDEX_PATH = ROOT / 'data' / 'interim' / 'retrieval_index'\nRUN_ID = f'offline_trace_validation_seed{SEED}_n{SAMPLE_SIZE}'\nOUT_DIR = ROOT / 'artifacts' / 'offline_trace_validation' / RUN_ID\nTRACE_PATH = OUT_DIR / 'trace_records.parquet'\nSUMMARY_PATH = OUT_DIR / 'diagnostic_summary.json'\nDETAIL_PATH = OUT_DIR / 'problem_diagnostics.json'\n\nREQUIRED_COLUMNS = ('id', 'problem')\n\nrandom.seed(SEED)\n\n\ndef need(condition, message):\n    if not condition:\n        raise AssertionError(message)\n\n\ndef sj(value):\n    return json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)\n\n\ndef model_to_data(value):\n    if value is None:\n        return None\n    if hasattr(value, 'model_dump'):\n        return value.model_dump(mode='json')\n    if hasattr(value, '__dict__'):\n        return dict(value.__dict__)\n    return value\n\n\ndef norm_entropy(probabilities):\n    values = [float(v) for v in dict(probabilities or {}).values() if float(v) > 0.0]\n    if not values:\n        return 0.0\n    total = sum(values)\n    need(total > 0.0, 'Probability mass must be positive.')\n    probs = [v / total for v in values]\n    entropy = -sum(p * math.log(p, 2) for p in probs if p > 0.0)\n    denom = math.log(len(probs), 2) if len(probs) > 1 else 1.0\n    return float(entropy / denom if denom else 0.0)\n\n\ndef prepare_output_dir(path, overwrite):\n    if path.exists():\n        if not overwrite:\n            raise FileExistsError(f'Output directory already exists: {path}')\n        shutil.rmtree(path)\n    path.mkdir(parents=True, exist_ok=True)\n\n\ndef load_dataset_rows(path):\n    need(path.exists(), f'Dataset not found: {path}')\n    need(path.stat().st_size > 0, f'Dataset is empty: {path}')\n    with path.open('r', encoding='utf-8', newline='') as handle:\n        reader = csv.DictReader(handle)\n        fieldnames = tuple(reader.fieldnames or ())\n        missing = [name for name in REQUIRED_COLUMNS if name not in fieldnames]\n        need(not missing, f'Dataset schema missing required columns: {missing}; found={fieldnames}')\n        rows = list(reader)\n    need(rows, 'Dataset has a header but no records.')\n    for index, row in enumerate(rows):\n        need(str(row.get('id', '')).strip(), f'Row {index} has empty id.')\n        need(str(row.get('problem', '')).strip(), f'Row {index} has empty problem text.')\n    return rows, fieldnames\n\n\ndef sample_rows(rows, sample_size, seed):\n    if len(rows) <= sample_size:\n        return list(rows)\n    picked = list(rows)\n    rng = random.Random(seed)\n    rng.shuffle(picked)\n    return sorted(picked[:sample_size], key=lambda row: str(row['id']))\n\n\ndef validate_parsed(parsed):\n    need(isinstance(parsed, ParsedProblem), f'Parser must return ParsedProblem, got {type(parsed)!r}.')\n    need(bool(parsed.problem_id), 'ParsedProblem.problem_id is empty.')\n    need(bool(str(parsed.raw_text).strip()), f'ParsedProblem.raw_text empty for {parsed.problem_id}.')\n    need(parsed.canonical_problem_graph is not None, f'Canonical problem graph missing for {parsed.problem_id}.')\n    constraint_texts = list(parsed.constraint_texts())\n    graph_payload = model_to_data(parsed.canonical_problem_graph)\n    need(bool(parsed.constraints or constraint_texts or parsed.knowns or parsed.unknowns), f'ParsedProblem is structurally empty for {parsed.problem_id}.')\n    need(bool(graph_payload), f'Constraint graph payload empty for {parsed.problem_id}.')\n    return {\n        'constraint_count': len(parsed.constraints),\n        'constraint_text_count': len(constraint_texts),\n        'known_count': len(parsed.knowns),\n        'unknown_count': len(parsed.unknowns),\n        'likely_archetypes': list(parsed.likely_archetypes),\n    }\n\n\ndef validate_route(route):\n    need(isinstance(route, RouteDecision), f'Router must return RouteDecision, got {type(route)!r}.')\n    problem_type_probs = dict(getattr(route, 'problem_type_probs', {}) or getattr(route, 'problem_type', {}) or {})\n    archetype_probs = dict(getattr(route, 'archetype_probs', {}) or getattr(route, 'archetypes', {}) or {})\n    for label, probs in (('problem_type_probs', problem_type_probs), ('archetype_probs', archetype_probs), ('operator_prior', dict(route.operator_prior or {}))):\n        need(probs, f'RouteDecision.{label} is empty for {route.problem_id}.')\n        total = sum(float(v) for v in probs.values())\n        need(total > 0.0, f'RouteDecision.{label} has non-positive mass for {route.problem_id}.')\n        need(all(0.0 <= float(v) <= 1.0 for v in probs.values()), f'RouteDecision.{label} has invalid probability values for {route.problem_id}.')\n        need(0.99 <= total <= 1.01, f'RouteDecision.{label} is not normalized for {route.problem_id}: {total}.')\n    budget = route.budget_plan\n    need(route.branch_budget > 0, f'Branch budget must be positive for {route.problem_id}.')\n    need(route.retrieval_depth >= 0, f'Retrieval depth must be non-negative for {route.problem_id}.')\n    need(getattr(budget, 'max_search_depth', 0) > 0, f'Max search depth must be positive for {route.problem_id}.')\n    need(getattr(budget, 'max_search_nodes', 0) > 0, f'Max search nodes must be positive for {route.problem_id}.')\n    return {\n        'difficulty': getattr(getattr(route, 'difficulty', None), 'value', str(getattr(route, 'difficulty', ''))),\n        'difficulty_score': float(route.difficulty_score),\n        'problem_type_probs': problem_type_probs,\n        'archetype_probs': archetype_probs,\n        'operator_prior': dict(route.operator_prior or {}),\n        'routing_entropy': norm_entropy(archetype_probs),\n        'budget_plan': model_to_data(budget),\n    }\n\n\ndef validate_state_graph(init_result):\n    graph = init_result.graph\n    root = init_result.root_node\n    need(graph.node_count >= 1, f'State graph is empty for {root.provenance.problem_id}.')\n    need(graph.get_node(root.node_id) is not None, f'Root node missing from graph for {root.provenance.problem_id}.')\n    need(bool(root.constraints or root.invariants or root.goals), f'Root node is missing constraints, invariants, and goals for {root.provenance.problem_id}.')\n    return {\n        'root_node_id': root.node_id,\n        'node_count': graph.node_count,\n        'edge_count': graph.edge_count,\n        'constraint_count': len(root.constraints),\n        'invariant_count': len(root.invariants),\n        'goal_count': len(root.goals),\n        'metadata': model_to_data(init_result.metadata),\n    }\n\n\ndef validate_branch_placeholders(route, parsed, root_node):\n    branch_state = BranchState.create(\n        route=route,\n        parsed_problem=parsed,\n        root_node=root_node,\n        metadata={'source': 'offline_trace_validation'},\n    )\n    need(hasattr(branch_state, 'symbolic_evidence'), f'BranchState.symbolic_evidence missing for {parsed.problem_id}.')\n    need(hasattr(branch_state, 'verifier_evidence'), f'BranchState.verifier_evidence missing for {parsed.problem_id}.')\n    need(isinstance(branch_state.active_symbolic_evidence(), tuple), f'BranchState active_symbolic_evidence must be tuple for {parsed.problem_id}.')\n    need(isinstance(branch_state.active_verifier_evidence(), tuple), f'BranchState active_verifier_evidence must be tuple for {parsed.problem_id}.')\n    return branch_state\n\n\ndef validate_prior(prior, problem_id):\n    normalized = dict(prior or {})\n    need(normalized, f'Operator prior is empty for {problem_id}.')\n    need(all(float(v) >= 0.0 for v in normalized.values()), f'Operator prior has negative mass for {problem_id}.')\n    total = sum(float(v) for v in normalized.values())\n    need(total > 0.0, f'Operator prior mass is non-positive for {problem_id}.')\n    need(0.99 <= total <= 1.01, f'Operator prior is not normalized for {problem_id}: {total}.')\n    return normalized\n\n\ndef failure_label(value):\n    if value is None:\n        return None\n    return getattr(value, 'value', str(value))\n\n\ndef top_items(score_map, limit=5):\n    ordered = sorted(dict(score_map or {}).items(), key=lambda item: (-float(item[1]), item[0]))\n    return ordered[:limit]\n\n\ndef make_stage_failure(problem_id, record_index, failed_stage, error, problem_snapshot):\n    return {\n        'record_type': 'stage_failure',\n        'record_index': record_index,\n        'run_id': RUN_ID,\n        'seed': SEED,\n        'problem_id': problem_id,\n        'branch_id': None,\n        'trace_id': None,\n        'trace_status': 'stage_failure',\n        'failed_stage': failed_stage,\n        'failure_labels': sj([failed_stage]),\n        'error': repr(error),\n        'step_count': 0,\n        'operator_count': 0,\n        'step_sequence': sj([]),\n        'operator_sequence': sj([]),\n        'symbolic_evidence': sj([]),\n        'verifier_evidence': sj([]),\n        'partial_trace_debug': sj({}),\n        'provenance': sj(problem_snapshot),\n    }\n

In [ ]:
prepare_output_dir(OUT_DIR, OVERWRITE_OUTPUTS)\n\nrows, schema = load_dataset_rows(DATASET_PATH)\nsampled_rows = sample_rows(rows, SAMPLE_SIZE, SEED)\n\nparser = ProblemParser()\nrouter = DualRouter()\nstate_initializer = StateGraphInitializer()\nembedder = MathEmbedder()\nindex = RetrievalIndexBuilder(embedder=embedder, index_path=str(INDEX_PATH))\nneed(index.load(), f'Retrieval index failed to load from {INDEX_PATH}.')\nretrieval_policy = RetrievalPolicy()\nretriever = TraceRetriever(embedder=embedder, index=index, policy=retrieval_policy)\noperator_library = OperatorLibrary()\nprior_shaper = OperatorPriorShaper(operator_library)\ncontroller = BranchController(\n    operator_library=operator_library,\n    prior_shaper=prior_shaper,\n    config=ControllerConfig(\n        deterministic=True,\n        self_consistency_samples=4,\n        frontier_width=4,\n        max_search_depth=4,\n        max_search_nodes=16,\n        retain_top_k=4,\n        repair_budget=1,\n        resample_budget=1,\n        critique_top_k=2,\n    ),\n)\n\ntrace_rows = []\nproblem_diagnostics = []\nstage_failure_counts = Counter()\nfailure_histogram = Counter()\noperator_histogram = Counter()\nrouting_entropies = []\nparse_failures = 0\ntrace_alignment_errors = 0\nbranch_success_count = 0\ngenerated_trace_count = 0\nrecord_index = 0\n\nfor sample_index, row in enumerate(sampled_rows):\n    problem_id = str(row['id']).strip()\n    problem_text = str(row['problem']).strip()\n    expected_answer = str(row.get('answer', '')).strip() or None\n    problem_snapshot = {\n        'run_id': RUN_ID,\n        'seed': SEED,\n        'sample_index': sample_index,\n        'dataset_path': str(DATASET_PATH),\n        'problem_id': problem_id,\n        'expected_answer': expected_answer,\n    }\n    diagnostic = dict(problem_snapshot)\n\n    try:\n        parsed = parser.parse_sync(problem_text, problem_id=problem_id)\n        diagnostic['parse'] = validate_parsed(parsed)\n    except Exception as exc:\n        parse_failures += 1\n        stage_failure_counts['parse'] += 1\n        failure_histogram['parse'] += 1\n        diagnostic['failed_stage'] = 'parse'\n        diagnostic['error'] = repr(exc)\n        trace_rows.append(make_stage_failure(problem_id, record_index, 'parse', exc, problem_snapshot))\n        record_index += 1\n        problem_diagnostics.append(diagnostic)\n        continue\n\n    try:\n        route = router.route(parsed)\n        route_info = validate_route(route)\n        routing_entropies.append(route_info['routing_entropy'])\n        diagnostic['route'] = route_info\n    except Exception as exc:\n        stage_failure_counts['routing'] += 1\n        failure_histogram['routing'] += 1\n        diagnostic['failed_stage'] = 'routing'\n        diagnostic['error'] = repr(exc)\n        trace_rows.append(make_stage_failure(problem_id, record_index, 'routing', exc, problem_snapshot))\n        record_index += 1\n        problem_diagnostics.append(diagnostic)\n        continue\n\n    try:\n        init_result = state_initializer.initialize(parsed, route)\n        diagnostic['state_init'] = validate_state_graph(init_result)\n        placeholder_state = validate_branch_placeholders(route, parsed, init_result.root_node)\n        diagnostic['branch_state_placeholders'] = {\n            'symbolic_placeholder_count': len(placeholder_state.active_symbolic_evidence()),\n            'verifier_placeholder_count': len(placeholder_state.active_verifier_evidence()),\n        }\n    except Exception as exc:\n        stage_failure_counts['state_init'] += 1\n        failure_histogram['state_init'] += 1\n        diagnostic['failed_stage'] = 'state_init'\n        diagnostic['error'] = repr(exc)\n        trace_rows.append(make_stage_failure(problem_id, record_index, 'state_init', exc, problem_snapshot))\n        record_index += 1\n        problem_diagnostics.append(diagnostic)\n        continue\n\n    try:\n        query = retriever.build_query(parsed, route, reasoning_state=init_result.root_node)\n        hits = retriever.retrieve_hits(query, route, top_k=max(1, int(route.retrieval_depth)))\n        retrieved_traces = [retrieval_policy.hit_to_trace(hit) for hit in hits]\n        operator_support = retrieval_policy.operator_support_signal(hits)\n        prior, prior_rows = prior_shaper.shape_priors(\n            node=init_result.root_node,\n            route=route,\n            retrieved_operator_hints=operator_support,\n        )\n        normalized_prior = validate_prior(prior, problem_id)\n        diagnostic['retrieval'] = {\n            'query': model_to_data(query),\n            'hit_count': len(hits),\n            'hit_trace_ids': [getattr(hit, 'trace_id', '') for hit in hits],\n            'top_operator_support': top_items(operator_support),\n        }\n        diagnostic['operator_priors'] = {\n            'top_prior': top_items(normalized_prior),\n            'policy_rows': [model_to_data(item) for item in prior_rows[:8]],\n        }\n    except Exception as exc:\n        stage_failure_counts['retrieval_operator_shaping'] += 1\n        failure_histogram['retrieval_operator_shaping'] += 1\n        diagnostic['failed_stage'] = 'retrieval_operator_shaping'\n        diagnostic['error'] = repr(exc)\n        trace_rows.append(make_stage_failure(problem_id, record_index, 'retrieval_operator_shaping', exc, problem_snapshot))\n        record_index += 1\n        problem_diagnostics.append(diagnostic)\n        continue\n\n    try:\n        result = controller.solve(\n            problem=parsed,\n            route=route,\n            root_node=init_result.root_node,\n            retrieved_traces=retrieved_traces,\n        )\n        need(result.all_branches, f'Branch controller generated no branches for {problem_id}.')\n        diagnostic['branch_controller'] = {\n            'branch_count': len(result.all_branches),\n            'surviving_branch_count': len(result.surviving_branches),\n            'stopped_reason': result.stopped_reason,\n            'selected_for_critique': list(result.selected_for_critique),\n        }\n    except Exception as exc:\n        stage_failure_counts['branch_controller'] += 1\n        failure_histogram['branch_controller'] += 1\n        diagnostic['failed_stage'] = 'branch_controller'\n        diagnostic['error'] = repr(exc)\n        trace_rows.append(make_stage_failure(problem_id, record_index, 'branch_controller', exc, problem_snapshot))\n        record_index += 1\n        problem_diagnostics.append(diagnostic)\n        continue\n\n    built_for_problem = 0\n    for branch in result.all_branches:\n        try:\n            trace = controller._to_branch_trace(branch)\n            steps = [\n                {\n                    'step_num': step.step_num,\n                    'description': step.description,\n                    'operator_used': step.operator_used,\n                    'symbolic_valid': step.symbolic_valid,\n                }\n                for step in trace.steps\n            ]\n            step_operators = [step['operator_used'] for step in steps if step['operator_used']]\n            operator_sequence = list(trace.operator_sequence or [])\n            need(operator_sequence == step_operators, f'TraceRecord operator alignment mismatch for {trace.branch_id}.')\n            failure_labels = [label for label in [failure_label(trace.failure_type)] if label]\n            if not diagnostic['retrieval']['hit_count']:\n                failure_labels.append('retrieval_miss')\n            if trace.failure_type is None:\n                branch_success_count += 1\n            else:\n                failure_histogram[failure_labels[0]] += 1\n            generated_trace_count += 1\n            for operator_name in operator_sequence:\n                operator_histogram[operator_name] += 1\n            trace_rows.append(\n                {\n                    'record_type': 'trace_record',\n                    'record_index': record_index,\n                    'run_id': RUN_ID,\n                    'seed': SEED,\n                    'problem_id': trace.problem_id,\n                    'branch_id': trace.branch_id,\n                    'trace_id': trace.branch_id,\n                    'trace_status': 'branch_success' if trace.failure_type is None else 'branch_failure',\n                    'failed_stage': None,\n                    'failure_labels': sj(failure_labels),\n                    'error': None,\n                    'expected_answer': expected_answer,\n                    'trace_answer': trace.answer,\n                    'trace_answer_canonical': trace.answer_canonical,\n                    'step_count': len(steps),\n                    'operator_count': len(operator_sequence),\n                    'step_sequence': sj(steps),\n                    'operator_sequence': sj(operator_sequence),\n                    'symbolic_evidence': sj([model_to_data(item) for item in branch.active_symbolic_evidence()]),\n                    'verifier_evidence': sj([model_to_data(item) for item in branch.active_verifier_evidence()]),\n                    'partial_trace_debug': sj({\n                        'phase': getattr(getattr(branch, 'phase', None), 'value', str(getattr(branch, 'phase', ''))),\n                        'preview_steps': [step['description'] for step in steps[:3]],\n                        'retrieval_hit_count': diagnostic['retrieval']['hit_count'],\n                    }),\n                    'symbolic_valid': bool(trace.symbolic_valid),\n                    'branch_score': float(trace.branch_score),\n                    'verifier_score': float(trace.verifier_score),\n                    'tool_consistency': float(trace.tool_consistency),\n                    'retrieval_used': bool(trace.retrieval_used),\n                    'archetype_used': trace.archetype_used,\n                    'provenance': sj({\n                        **problem_snapshot,\n                        'route': {\n                            'difficulty': diagnostic['route']['difficulty'],\n                            'difficulty_score': diagnostic['route']['difficulty_score'],\n                            'archetype_probs': diagnostic['route']['archetype_probs'],\n                            'budget_plan': diagnostic['route']['budget_plan'],\n                        },\n                        'retrieval_query': diagnostic['retrieval']['query'],\n                        'retrieval_hit_ids': diagnostic['retrieval']['hit_trace_ids'],\n                    }),\n                }\n            )\n            record_index += 1\n            built_for_problem += 1\n        except Exception as exc:\n            trace_alignment_errors += 1\n            stage_failure_counts['trace_build'] += 1\n            failure_histogram['trace_build'] += 1\n            trace_rows.append(make_stage_failure(problem_id, record_index, 'trace_build', exc, problem_snapshot))\n            record_index += 1\n\n    need(built_for_problem > 0, f'No TraceRecord rows were built for {problem_id}.')\n    diagnostic['trace_build'] = {'trace_count': built_for_problem}\n    problem_diagnostics.append(diagnostic)\n\nstage_failures_total = sum(stage_failure_counts.values())\nparse_failure_rate = parse_failures / len(sampled_rows)\nrouting_entropy_mean = sum(routing_entropies) / len(routing_entropies) if routing_entropies else 0.0\nbranch_success_rate = branch_success_count / generated_trace_count if generated_trace_count else 0.0\nstage_fails_most_often = stage_failure_counts.most_common(1)[0][0] if stage_failure_counts else 'none_detected'\nweakest_upstream_module = stage_fails_most_often if stage_fails_most_often in {'parse', 'routing', 'state_init', 'retrieval_operator_shaping'} else 'none_detected'\ntraces_usable_for_mining = bool(generated_trace_count and branch_success_rate >= 0.25 and trace_alignment_errors == 0)\n\nsummary = {\n    'run_id': RUN_ID,\n    'seed': SEED,\n    'dataset_path': str(DATASET_PATH),\n    'dataset_schema': list(schema),\n    'sample_size_requested': SAMPLE_SIZE,\n    'sample_size_executed': len(sampled_rows),\n    'retrieval_index_path': str(INDEX_PATH),\n    'parse_failure_rate': parse_failure_rate,\n    'routing_entropy': routing_entropy_mean,\n    'branch_success_rate': branch_success_rate,\n    'failure_type_histogram': dict(sorted(failure_histogram.items())),\n    'stage_failure_counts': dict(sorted(stage_failure_counts.items())),\n    'stage_failures_total': stage_failures_total,\n    'operator_usage_patterns': dict(operator_histogram.most_common(20)),\n    'trace_alignment_errors': trace_alignment_errors,\n    'generated_trace_count': generated_trace_count,\n    'stage_fails_most_often': stage_fails_most_often,\n    'traces_usable_for_mining': traces_usable_for_mining,\n    'weakest_upstream_module': weakest_upstream_module,\n}\n\npd.DataFrame(trace_rows).to_parquet(TRACE_PATH, index=False)\nSUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')\nDETAIL_PATH.write_text(json.dumps(problem_diagnostics, indent=2, ensure_ascii=True, sort_keys=True, default=str), encoding='utf-8')\n\nsummary\n

In [ ]:
diagnostic_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))\ntrace_frame = pd.read_parquet(TRACE_PATH)\n\nfinal_report = {\n    '1_which_stage_fails_most_often': diagnostic_summary['stage_fails_most_often'],\n    '2_whether_traces_are_usable_for_mining': diagnostic_summary['traces_usable_for_mining'],\n    '3_weakest_upstream_module': diagnostic_summary['weakest_upstream_module'],\n}\n\nprint(json.dumps(final_report, indent=2, ensure_ascii=True, sort_keys=True))\ntrace_frame.head(10)\n